# Урок 8. Представление чисел в памяти компьютера

10 класс · I четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [← Урок 7](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-07.ipynb) · [Урок 9 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-09.ipynb)

---

Разрядность, диапазоны, переполнение. Дополнительный код. Числа с плавающей запятой и потеря точности. Сравнение вещественных чисел.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 10А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="10-08", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### От абстракции к железу

До сих пор числа для нас были математическими объектами. Сейчас
спустимся на уровень памяти: как конкретно число лежит в ячейках,
что происходит на границах диапазона и почему вычисления бывают
неточными.

### Целые числа: разрядная сетка

Под целое число отводится фиксированное количество бит.
В беззнаковом представлении:

$$\text{диапазон} = [0,\; 2^n - 1]$$

В знаковом (дополнительный код):

$$\text{диапазон} = [-2^{n-1},\; 2^{n-1} - 1]$$

| Тип | Бит | Беззнаковый | Знаковый |
|---|---|---|---|
| byte | 8 | 0 … 255 | −128 … 127 |
| short | 16 | 0 … 65 535 | −32 768 … 32 767 |
| int | 32 | 0 … 4 294 967 295 | −2 147 483 648 … 2 147 483 647 |
| long | 64 | 0 … ≈1,8·10¹⁹ | ≈−9,2·10¹⁸ … 9,2·10¹⁸ |

### Дополнительный код: напоминание и обоснование

Отрицательное число −x в n разрядах записывается как $2^n - x$.

Практическое следствие: вычитание сводится к сложению. Процессору
не нужен отдельный блок вычитания, достаточно сумматора и схемы
инверсии. Это экономия транзисторов, которая когда-то была критичной,
а сейчас просто стала стандартом.

### Переполнение и его последствия

При выходе за границу диапазона старшие разряды отбрасываются.
В беззнаковой арифметике 255 + 1 даёт 0, в знаковой 127 + 1 даёт −128.

Второй случай опаснее: **прибавили положительное число,
а получили отрицательное**. Проверка `if a + b > 0` может дать
неверный результат, и найти такую ошибку тяжело.

Реальные последствия переполнения:

* **Ariane 5, 1996** — при преобразовании 64-разрядного вещественного
  в 16-разрядное целое произошло переполнение, ракета разрушилась
  через 37 секунд после старта.
* **Проблема 2038 года** — во многих старых системах время хранится
  как 32-разрядное число секунд с 1970 года. 19 января 2038 года оно
  переполнится. Современные системы уже перешли на 64 разряда,
  а вот старые устройства и программы остались.

### Вещественные числа: формат IEEE 754

```
  ┌─┬────────────┬─────────────────────────────────┐
  │s│  порядок   │            мантисса             │
  └─┴────────────┴─────────────────────────────────┘
   1     11 бит              52 бита            (double, 64 бита)
   1      8 бит              23 бита            (float, 32 бита)
```

Число восстанавливается как

$$(-1)^s \cdot 1{,}M \cdot 2^{E - \text{смещение}}$$

Единица перед мантиссой подразумевается и не хранится — это даёт
лишний бит точности бесплатно.

### Что из этого следует

**Диапазон огромен, точность ограничена.** Тип double хранит числа
до $10^{308}$, но значащих цифр всего около 16. Число $10^{20} + 1$
в нём неотличимо от $10^{20}$.

**Не все дроби представимы.** В двоичной системе точно представимы
только дроби со знаменателем — степенью двойки: 0,5, 0,25, 0,125.
Дробь 0,1 — бесконечная периодическая, и хранится приближённо.

**Ошибки накапливаются.** Сложение миллиона приближённых значений
даёт заметно неточный результат.

### Правила безопасной работы

| Никогда | Вместо этого |
|---|---|
| `if a == b` для дробных | `if abs(a - b) < eps` |
| хранить деньги в float | хранить копейки в int |
| накапливать сумму дробных в цикле | суммировать целые, делить в конце |
| сравнивать через вычитание больших чисел | сравнивать напрямую |

### Машинный эпсилон

Наименьшее число, которое ещё различимо при добавлении к единице,
называется **машинным эпсилоном**. Для double это примерно
$2{,}2 \cdot 10^{-16}$. Всё, что меньше, при сложении с единицей
просто теряется.

## Смотрим, как это работает

### Пример 1. Границы типов

In [ ]:
print(f"{'разрядов':>10} {'беззнаковый максимум':>24} {'знаковый диапазон':>32}")
for n in [8, 16, 32, 64]:
    без = 2 ** n - 1
    от, до = -2 ** (n - 1), 2 ** (n - 1) - 1
    print(f"{n:>10} {без:>24,} {f'{от:,} … {до:,}':>32}")

Обратите внимание на 32-разрядный знаковый максимум: 2 147 483 647.
Это число вы наверняка видели — оно всплывает в играх и программах
как «предел» очков или денег.

### Пример 2. Переполнение со знаком

In [ ]:
def в_знаковую_ячейку(число, разрядов=8):
    предел = 2 ** разрядов
    остаток = число % предел
    половина = предел // 2
    return остаток - предел if остаток >= половина else остаток


print("Восьмиразрядная знаковая ячейка:")
for число in [125, 126, 127, 128, 129, 255, 256]:
    print(f"  кладём {число:>4} → получаем {в_знаковую_ячейку(число):>5}")

Переход от 127 к 128 даёт скачок с +127 на −128. Именно этот эффект
и называется переполнением со знаком: число «перевалило через край»
и вынырнуло с другой стороны диапазона.

### Пример 3. Точность вещественных

In [ ]:
import math

print("Какие дроби представимы точно:")
for дробь in [0.5, 0.25, 0.125, 0.1, 0.2, 0.3]:
    точно = дробь == float(f"{дробь:.20f}".rstrip("0"))
    print(f"  {дробь}: {дробь:.20f}")

print()
print("Машинный эпсилон:")
эпсилон = 1.0
while 1.0 + эпсилон / 2 != 1.0:
    эпсилон /= 2
print(f"  {эпсилон}")
print(f"  для сравнения, из библиотеки: {2.220446049250313e-16}")

print()
print("Потеря значащих цифр:")
большое = 1e20
print(f"  1e20 + 1 == 1e20 ? {большое + 1 == большое}")
print(f"  1e15 + 1 == 1e15 ? {1e15 + 1 == 1e15}")

Первые три дроби записаны точно — у них знаменатели 2, 4 и 8.
Остальные имеют «хвост» в двадцатом знаке.

А последние строки показывают потерю значащих цифр: к числу $10^{20}$
единицу прибавить уже невозможно, она не помещается в мантиссу.

### Пример 4. Накопление ошибки и как его избежать

In [ ]:
# Плохо: накапливаем дробные
плохо = 0.0
for _ in range(1000000):
    плохо += 0.1

# Хорошо: накапливаем целые, делим в конце
хорошо = 0
for _ in range(1000000):
    хорошо += 1
хорошо = хорошо * 0.1

print(f"Ожидаем:         100000.0")
print(f"Накопление float: {плохо}")
print(f"Накопление int:   {хорошо}")
print(f"Ошибка первого способа: {abs(плохо - 100000)}")

Разница накопилась до заметной величины. Второй способ точен, потому
что целые числа складываются без потерь, а единственное умножение
в конце вносит только одну ошибку округления вместо миллиона.

## Пробуем сами

### Задача 1. Проверка на переполнение

Поместится ли число в знаковую ячейку заданной разрядности?
Верните `True` или `False`.

In [ ]:
def помещается(число, разрядов):
    return ...

In [ ]:
si.check("1", помещается, [
    ((127, 8), True),
    ((128, 8), False),
    ((-128, 8), True),
    ((-129, 8), False),
    ((0, 8), True),
    ((2147483647, 32), True),
])

### Задача 2. Результат переполнения

Верните значение, которое окажется в **знаковой** ячейке заданной
разрядности, если положить туда число.

Опирайтесь на пример 2.

In [ ]:
def в_ячейку(число, разрядов):
    return ...

In [ ]:
si.check("2", в_ячейку, [
    ((127, 8), 127),
    ((128, 8), -128),
    ((255, 8), -1),
    ((256, 8), 0),
    ((-129, 8), 127),
])

### Задача 3. Безопасное сравнение

Сравните два вещественных числа с относительной точностью:
считайте их равными, если разница не превышает `1e-9` от большего
по модулю из них.

Отдельно обработайте случай, когда оба равны нулю.

In [ ]:
def равны(а, б):
    return ...

In [ ]:
si.check("3", равны, [
    ((0.1 + 0.2, 0.3), True),
    ((1.0, 1.0), True),
    ((0.0, 0.0), True),
    ((1.0, 1.1), False),
    ((1e20, 1e20 + 1), True),
])

## Домашнее задание

### Домашнее задание 1. Дополнительный код

Верните представление числа в дополнительном коде заданной
разрядности — строкой из нулей и единиц.

In [ ]:
def доп_код(число, разрядов):
    return ...

In [ ]:
si.check("дз1", доп_код, [
    ((5, 8), "00000101"),
    ((-5, 8), "11111011"),
    ((-1, 16), "1111111111111111"),
    ((0, 4), "0000"),
    ((-128, 8), "10000000"),
])

### Домашнее задание 2. Деньги в копейках

Напишите функцию, которая суммирует список денежных сумм в рублях
**без потери точности** и возвращает результат в рублях.

Переводите каждую сумму в целые копейки, складывайте целыми,
делите на 100 только в конце.

In [ ]:
def сумма_денег(суммы):
    return ...

In [ ]:
si.check("дз2", сумма_денег, [
    ([0.1, 0.2], 0.3),
    ([0.01] * 100, 1.0),
    ([19.99, 0.01], 20.0),
    ([], 0.0),
])

### Домашнее задание 3. Наименьшая разрядность

Верните минимальное количество разрядов, необходимое для хранения
числа в знаковом виде.

`разрядность(127)` → `8`, `разрядность(128)` → `9`

Перебирайте разрядность от 1 и проверяйте, помещается ли число.

In [ ]:
def разрядность(число):
    return ...

In [ ]:
si.check("дз3", разрядность, [
    (127, 8),
    (128, 9),
    (-128, 8),
    (0, 1),
    (-1, 1),
    (2147483647, 32),
])

---

### Проверьте на своём компьютере

Запустите в отдельной ячейке:

```python
import sys
print(sys.float_info)
```

Вы увидите точные характеристики вещественных чисел на вашей машине:
максимум, минимум, машинный эпсилон, количество разрядов мантиссы.
Все они — прямое следствие формата, который мы разобрали.

А заодно обратите внимание на любопытную особенность Python:
**целые числа в нём неограниченны**. Попробуйте `2 ** 1000` — получите
точный ответ. Python сам выделяет столько памяти, сколько нужно,
и переполнения целых в нём не бывает. Это редкость: в большинстве
языков всё устроено так, как описано в теории.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 7](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-07.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 9 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-09.ipynb)